In [2]:
import ase
import numpy as np
import pyscf
import time
import os

In [1]:
from pyscf import gto, dft
from pyscf.scf import hf
hf.MUTE_CHKFILE = True

#mol = gto.M(atom='O  0  0  0.1184; H  0,  0.7532, -0.4735; H 0,  -0.7532, -0.4735 ', basis='def2svp')
#mf = dft.RKS(mol)
#mf.chkfile=False
#mf.xc = 'pbe'
#mf.kernel()
#g = mf.nuc_grad_method()
#g.kernel()
data = np.load('datasets/ethanol_dft_test.npy', allow_pickle=True).item()

atom_types = data['atom_types']

print(len(data['positions']))
save_path = 'datasets/ethanol_pyscf_def2svp_dft_f_test.npy'
if os.path.exists(save_path):
    results = list(np.load(save_path, allow_pickle=True))
else:
    results = []
print('results len', len(results))
for i in range(len(results), len(data['positions'])):
    print('calc', i)
    start = time.time()
    pos = data['positions'][i]
    atom = []
    for j in range(len(atom_types)):
        atom.append((atom_types[j], pos[j, :])) 
    mol = gto.M(atom=atom, basis='def2svp')
    #print(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile=False
    mf.xc = 'pbe'
    mf.kernel()
    g = mf.nuc_grad_method()
    forces = g.grad()
    print('elapsed', time.time() - start)
    #print(mfs[i].mo_coeff)
    res = []
    res.append(mol.pack())
    calc_dict = {}
    print('mo occ', mf.mo_occ)
    calc_dict['mo_coeff'] = mf.mo_coeff
    calc_dict['mo_occ'] = mf.mo_occ
    calc_dict['energy'] = mf.e_tot
    calc_dict['forces'] = forces
    res.append(calc_dict)
    results.append(res)
    
    if i%100 == 0:
        np.save(save_path, results, allow_pickle=True)
np.save(save_path, results, allow_pickle=True)

/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


NameError: name 'np' is not defined